In [ ]:
%pip install scikit-learn -q

import os
import json
import torch
import transformers

# Verify scikit-learn installation
try:
    from sklearn.model_selection import train_test_split
    sklearn_installed = True
except ImportError:
    sklearn_installed = False

print(f"Transformers version: {transformers.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"Sklearn Installed: {sklearn_installed}")

# Define project directories
PROJECT_DIR = os.getcwd()
DATA_DIR = os.path.join(PROJECT_DIR, "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
MODEL_DIR = os.path.join(PROJECT_DIR, "pythia_3bin_base")
OUTPUT_DIR = os.path.join(PROJECT_DIR, "pythia_70m_trained")

# Create necessary folder structure
for folder in [DATA_DIR, PROCESSED_DIR, MODEL_DIR, OUTPUT_DIR]:
    os.makedirs(folder, exist_ok=True)

# Final environment check
if sklearn_installed and torch.cuda.is_available():
    print("Environment Verified: Everything is ready. Please provide the next instruction.")
elif not sklearn_installed:
    print("Error: Scikit-learn is still not detected. You might need to restart the kernel and run Cell 1 again.")
else:
    print("Warning: GPU not detected. Training will be slow.")

Note: you may need to restart the kernel to use updated packages.
Transformers version: 5.7.0
PyTorch version: 2.11.0+cu128
GPU Available: True
Sklearn Installed: True
Environment Verified: Everything is ready. Please provide the next instruction.


In [ ]:
import sys
import os

# Clone the research repository if it doesn't exist
if not os.path.exists("personalized-gen"):
    print("Cloning the repository now")
    !git clone https://github.com/balhafni/personalized-gen.git
else:
    print("Repository already exists")

PROJECT_DIR = os.getcwd()
eval_path = os.path.join(PROJECT_DIR, "personalized-gen", "eval")

# Add the evaluation directory to system path for module imports
if eval_path not in sys.path:
    sys.path.append(eval_path)
    print("Added the evaluation path to system")

# Verify presence of the training script
script_path = os.path.join(PROJECT_DIR, "personalized-gen", "run_clm.py")
if os.path.exists(script_path):
    print("Verification Successful: The training script is found. Please provide the next instruction.")
else:
    print("Error: The training script was not found in the cloned repository.")

Cloning the repository now
Cloning into 'personalized-gen'...
remote: Enumerating objects: 281, done.
remote: Counting objects: 100% (281/281), done.
remote: Compressing objects: 100% (213/213), done.
remote: Total 281 (delta 95), reused 207 (delta 46), pack-reused 0 (from 0)
Receiving objects: 100% (281/281), 24.71 MiB | 11.57 MiB/s, done.
Resolving deltas: 100% (95/95), done.
Added the evaluation path to system
Verification Successful: The training script is found. Please provide the next instruction.


In [ ]:
import json
import os
from datasets import load_dataset, concatenate_datasets

# Define the bins configuration for linguistic features
bins_config = {
    "author_atts_bins": {
        "tokens": [0, 50, 150],
        "sents": [0, 3, 7],
        "POS_NOUN": [0, 0.2, 0.4],
        "POS_VERB": [0, 0.1, 0.2],
        "fkgl": [0, 6, 12],
        "domain": ["Blogs", "IMDb"]
    }
}

# Save the bins configuration to a JSON file
bins_path = os.path.join(DATA_DIR, "bins.json")
with open(bins_path, "w") as f:
    json.dump(bins_config, f)

# Helper function to add standardized metadata to dataset examples
def add_metadata(example, domain_name):
    return {
        "text": example["text"],
        "author": f"{domain_name}_user",
        "domain": domain_name,
        "title": "post_or_review"
    }

print("Loading two datasets with 15,000 samples each")

try:
    # Load and process Blogs dataset
    print("Loading Blogs dataset...")
    blog_data = load_dataset("tasksource/blog_authorship_corpus", split="train", revision="main").shuffle(seed=42).select(range(15000))
    blog_data = blog_data.remove_columns([c for c in blog_data.column_names if c != "text"]).map(lambda x: add_metadata(x, "Blogs"))
    print("Blogs dataset loaded successfully")

    # Load and process IMDb dataset
    print("Loading IMDb dataset...")
    imdb_data = load_dataset("imdb", split="train").shuffle(seed=42).select(range(15000))
    imdb_data = imdb_data.remove_columns([c for c in imdb_data.column_names if c != "text"]).map(lambda x: add_metadata(x, "IMDb"))
    print("IMDb dataset loaded successfully")

    # Combine and shuffle datasets
    combined_dataset = concatenate_datasets([blog_data, imdb_data]).shuffle(seed=42)

    # Save the combined raw data to a JSONL file
    raw_output_path = os.path.join(DATA_DIR, "raw_30k.json")
    with open(raw_output_path, "w") as f:
        for entry in combined_dataset:
            if entry["text"] and len(entry["text"].split()) >= 5:
                f.write(json.dumps(entry) + "\n")

    print(f"Success: {len(combined_dataset)} total samples saved to raw_30k.json")
    print("Please provide the next instruction.")

except Exception as e:
    print(f"Error during data loading: {e}")

Loading two datasets with 15,000 samples each
Loading Blogs dataset...


Map:   0%|          | 0/15000 [00:00<?, ? examples/s]

Blogs dataset loaded successfully
Loading IMDb dataset...


Map:   0%|          | 0/15000 [00:00<?, ? examples/s]

IMDb dataset loaded successfully
Success: 30000 total samples saved to raw_30k.json
Please provide the next instruction.


In [ ]:
import json
import os
import sys

PROJECT_DIR = os.getcwd()
eval_path = os.path.join(PROJECT_DIR, "personalized-gen", "eval")
if eval_path not in sys.path:
    sys.path.append(eval_path)

from annotate import annotate_data_author
from utils.discretize import discretize_attributes

# Load the bins configuration
bins_path = os.path.join(DATA_DIR, "bins.json")
with open(bins_path, "r") as f:
    bins_config = json.load(f)

# Load raw data for linguistic annotation
raw_data_path = os.path.join(DATA_DIR, "raw_30k.json")
print("Loading raw data for annotation")
with open(raw_data_path, "r") as f:
    data_to_annotate = [json.loads(line) for line in f]

print(f"Starting Linguistic Annotation on {len(data_to_annotate)} samples")
print("Note: This process involves POS tagging and may take some time")

try:
    bins_info = bins_config['author_atts_bins']
    # Perform linguistic feature extraction
    annotated_data = annotate_data_author(data=data_to_annotate, bins=bins_info, rst_parses=None)

    # Discretize continuous linguistic attributes into bins
    final_processed_data = discretize_attributes(data=annotated_data, bins=bins_info, mode='author_atts')

    # Save the final annotated dataset
    master_binned_path = os.path.join(PROCESSED_DIR, "master_binned.json")
    with open(master_binned_path, "w") as f:
        for entry in final_processed_data:
            f.write(json.dumps(entry) + "\n")

    print("Success: Annotation complete")
    print(f"Final dataset saved to: {master_binned_path}")
    print("Please provide the next instruction.")

except Exception as e:
    print(f"Error during annotation process: {e}")

/home/ubuntu/personalized-gen/eval/utils/data_utils.py:82: SyntaxWarning: invalid escape sequence '\s'
  rst_tree = re.sub('\s+ ', ' ', rst_tree.replace('(', ' ').replace(')', ' ')).strip()


Loading raw data for annotation
Starting Linguistic Annotation on 29473 samples
Note: This process involves POS tagging and may take some time
Done!
Success: Annotation complete
Final dataset saved to: /home/ubuntu/data/processed/master_binned.json
Please provide the next instruction.


In [ ]:
import json
import os
from sklearn.model_selection import train_test_split

master_binned_path = os.path.join(PROCESSED_DIR, "master_binned.json")

print("Loading master annotated dataset")
with open(master_binned_path, "r") as f:
    all_data = [json.loads(line) for line in f]

# Split data into training (80%) and a temporary set (20%)
train_data, temp_data = train_test_split(all_data, test_size=0.2, random_state=42)

# Split temporary set into validation (10%) and test (10%)
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

# Helper function to save data splits to JSONL files
def save_split_file(filename, data_list):
    file_path = os.path.join(PROCESSED_DIR, filename)
    with open(file_path, "w") as f:
        for entry in data_list:
            f.write(json.dumps(entry) + "\n")
    print(f"Saved {len(data_list)} samples to {file_path}")

# Save the generated train, validation, and test datasets
save_split_file("train.jsonl", train_data)
save_split_file("val.jsonl", val_data)
save_split_file("test.jsonl", test_data)

print("Success: Data splitting is complete")
print("Training and Validation files are now in separate paths")
print("Please provide the next instruction.")

Loading master annotated dataset
Saved 23578 samples to /home/ubuntu/data/processed/train.jsonl
Saved 2947 samples to /home/ubuntu/data/processed/val.jsonl
Saved 2948 samples to /home/ubuntu/data/processed/test.jsonl
Success: Data splitting is complete
Training and Validation files are now in separate paths
Please provide the next instruction.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import os

os.makedirs(MODEL_DIR, exist_ok=True)

print("Downloading base Pythia-70M model and tokenizer")
# Load pre-trained Pythia-70M model and its tokenizer
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/pythia-70m-deduped")
model = AutoModelForCausalLM.from_pretrained("EleutherAI/pythia-70m-deduped")

# Define custom control tokens for linguistic attributes
control_tokens = [
    "<|tokens:0|>", "<|tokens:1|>", "<|tokens:2|>",
    "<|sents:0|>", "<|sents:1|>", "<|sents:2|>",
    "<|POS_NOUN:0|>", "<|POS_NOUN:1|>", "<|POS_NOUN:2|>",
    "<|POS_VERB:0|>", "<|POS_VERB:1|>", "<|POS_VERB:2|>",
    "<|fkgl:0|>", "<|fkgl:1|>", "<|fkgl:2|>"
]

print("Adding new control tokens to vocabulary")
# Add new tokens to the tokenizer and resize model embeddings
tokenizer.add_tokens(control_tokens)
model.resize_token_embeddings(len(tokenizer))

# Set padding token to end-of-sequence token if not already defined
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = model.config.eos_token_id

print(f"Saving modified model and tokenizer to {MODEL_DIR}")
# Save the modified model and tokenizer locally
tokenizer.save_pretrained(MODEL_DIR)
model.save_pretrained(MODEL_DIR)

print("Success: Model prepared with 15 control tokens")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Adding new control tokens to vocabulary
Saving modified model and tokenizer to /home/ubuntu/pythia_3bin_base


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Success: Model prepared with 15 control tokens
Please provide the next instruction.


In [ ]:
import shutil
import os

source_script = os.path.join(PROJECT_DIR, "personalized-gen", "run_clm.py")
target_script = os.path.join(PROJECT_DIR, "run_clm.py")

print("Copying run_clm.py to the main directory")
# Copy the training script from the cloned repository
shutil.copy(source_script, target_script)

with open(target_script, "r") as f:
    script_code = f.read()

print("Applying compatibility patches to the script")

# Apply patches for compatibility with current library versions
script_code = script_code.replace("is_torch_tpu_available", "is_torch_xla_available")
script_code = script_code.replace(", send_example_telemetry", "")
script_code = script_code.replace("send_example_telemetry(", "# send_example_telemetry(")
script_code = script_code.replace("use_auth_token", "token")
script_code = script_code.replace("tokenizer=tokenizer", "processing_class=tokenizer")
script_code = script_code.replace("MyCollator(processing_class=tokenizer", "MyCollator(tokenizer=tokenizer")
script_code = script_code.replace("and not training_args.overwrite_output_dir", "")

# Write the patched script back to the file
with open(target_script, "w") as f:
    f.write(script_code)

if os.path.exists(target_script):
    print("Success: run_clm.py is prepared and patched")
else:
    print("Error: Failed to copy or patch the training script.")

Copying run_clm.py to the main directory
Applying compatibility patches to the script
Success: run_clm.py is prepared and patched


In [ ]:
%pip install --upgrade accelerate peft transformers scikit-learn -q

import os
import accelerate
import transformers
# Check if PEFT library is installed and ready
try:
    import peft
    peft_status = "Ready"
except ImportError:
    peft_status = "Not Found"

print(f"Accelerate version: {accelerate.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"PEFT status: {peft_status}")

# Provide instructions if PEFT is not ready, usually requires kernel restart
if peft_status == "Ready":
    print("Success: All libraries are now recognized by the kernel.")
else:
    print("Action Required: Please go to 'Kernel' -> 'Restart' in the top menu, then run Cell 8.")

Note: you may need to restart the kernel to use updated packages.
Accelerate version: 1.13.0
Transformers version: 5.7.0
PEFT status: Ready
Success: All libraries are now recognized by the kernel.


In [ ]:
import json
import nltk
import re

# Download necessary NLTK data for tokenization and POS tagging
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

def preprocess_and_add_tokens(input_file, output_file):
    processed_data = []
    print("Starting token prepending for Uzma's dataset...")

    with open(input_file, 'r') as f:
        for line in f:
            item = json.loads(line)
            raw_text = item.get('text', "")

            # Calculate sentence count
            sents = [s for s in re.split(r'[.!?]+', raw_text) if len(s.strip()) > 3]
            sents_count = len(sents)

            # Calculate noun count using NLTK POS tagging
            words = nltk.word_tokenize(raw_text)
            tags = nltk.pos_tag(words)
            noun_count = len([w for w, t in tags if t.startswith('NN')])

            # Map counts to 3-bin categories (example logic)
            if sents_count <= 2: s_bin = 1
            elif sents_count <= 5: s_bin = 2
            else: s_bin = 3

            if noun_count <= 5: n_bin = 1
            elif noun_count <= 15: n_bin = 2
            else: n_bin = 3

            # Prepend control tokens to the text
            control_tokens = f"<|sents:{s_bin}|> <|pos_noun:{n_bin}|>"
            new_text = f"{control_tokens} {raw_text}"

            processed_data.append({"text": new_text})

    # Save the processed data with prepended tokens
    with open(output_file, 'w') as f:
        for entry in processed_data:
            f.write(json.dumps(entry) + '\n')

    print(f"Success! New training file created at: {output_file}")

# Execute the preprocessing for the training file
preprocess_and_add_tokens("./data/processed/train.json", "./data/processed/train_with_tokens.json")

Starting token prepending for Uzma's dataset...
Success! New training file created at: ./data/processed/train_with_tokens.json


In [ ]:
import json

file_to_check = "./data/processed/train_with_tokens.json"

try:
    with open(file_to_check, 'r') as f:
        print(f"{'='*20} DATA AUDIT {'='*20}\n")
        # Print the first 3 samples to audit the prepended tokens
        for i in range(3):
            line = f.readline()
            if not line:
                break
            item = json.loads(line)
            print(f"Sample {i+1}:")
            print(f"{item['text'][:200]}...")
            print("-" * 50)
except FileNotFoundError:
    print("Error: 'train_with_tokens.json' abhi tak generate nahi hui.")

==================== DATA AUDIT ====================

Sample 1:
<|sents:3|> <|pos_noun:3|> Just got home from the VERY long morning I spent at the DMV.  I took my Mom's advice and blinked a lot when I looked into the vision test machine, which somehow worked (thou...
--------------------------------------------------
Sample 2:
<|sents:3|> <|pos_noun:3|> I'm smiling. The results of the search engine optimization efforts I did for a website are coming in...and I have pretty decent positions on common keywords. Can you say  ur...
--------------------------------------------------
Sample 3:
<|sents:3|> <|pos_noun:3|> While I can't deny that his movies are often entertaining, I have always personally felt that Martin Scorsese is just a little overrated in his abilities. His use of flashy ...
--------------------------------------------------


In [ ]:
import json
import random

input_file = "./data/processed/train_with_tokens.json"
train_output = "./data/processed/train.json"
val_output = "./data/processed/val.json"

def split_dataset(file_path, split_ratio=0.8):
    with open(file_path, 'r') as f:
        lines = f.readlines()

    # Shuffle the dataset lines for random splitting
    random.shuffle(lines)

    # Determine split index and divide into train and validation sets
    split_index = int(len(lines) * split_ratio)
    train_data = lines[:split_index]
    val_data = lines[split_index:]

    # Save the training data
    with open(train_output, 'w') as f:
        for line in train_data:
            f.write(line)

    # Save the validation data
    with open(val_output, 'w') as f:
        for line in val_data:
            f.write(line)

    print(f"✅ Splitting Complete!")
    print(f"Total Samples: {len(lines)}")
    print(f"Training Samples: {len(train_data)}")
    print(f"Validation Samples: {len(val_data)}")

split_dataset(input_file)

✅ Splitting Complete!
Total Samples: 23578
Training Samples: 18862
Validation Samples: 4716


In [ ]:
import json
import os

model_dir = "./pythia_3bin_base"
config_files = ["config.json", "generation_config.json"]

# Iterate through model configuration files
for file_name in config_files:
    path = os.path.join(model_dir, file_name)
    if os.path.exists(path):
        with open(path, "r") as f:
            data = json.load(f)

        # Force 'torch_dtype' and 'dtype' to 'float32' to prevent issues
        data["torch_dtype"] = "float32"
        if "dtype" in data:
            data["dtype"] = "float32"

        with open(path, "w") as f:
            json.dump(data, f, indent=2)
        print(f"Fixed {file_name} to use float32")

print("full precision")

Fixed config.json to use float32
Fixed generation_config.json to use float32
full precision


In [ ]:
import os
import shutil

MODEL_OUTPUT = "./pythia_70m_research_final"
# Clean up previous model output directory for a fresh training run
if os.path.exists(MODEL_OUTPUT):
    print(f"Cleaning up {MODEL_OUTPUT} for a fresh start...")
    shutil.rmtree(MODEL_OUTPUT)

# Define paths for the input model and training/validation data
MODEL_INPUT = "./pythia_3bin_base"
TRAIN_FILE = "./data/processed/train.json"
VAL_FILE = "./data/processed/val.json"

# Set training hyperparameters
EPOCHS = 5
LEARNING_RATE = 5e-5

print(f"🚀 Starting FRESH training in {MODEL_OUTPUT} for {EPOCHS} epochs...")

# Execute the training script with specified arguments
!python3 run_clm.py \
    --model_name_or_path "{MODEL_INPUT}" \
    --train_file "{TRAIN_FILE}" \
    --validation_file "{VAL_FILE}" \
    --do_train \
    --do_eval \
    --block_size 512 \
    --learning_rate {LEARNING_RATE} \
    --num_train_epochs {EPOCHS} \
    --warmup_steps 150 \
    --weight_decay 0.05 \
    --per_device_train_batch_size 32 \
    --gradient_accumulation_steps 1 \
    --lr_scheduler_type "cosine" \
    --logging_steps 10 \
    --eval_strategy "steps" \
    --eval_steps 200 \
    --save_strategy "steps" \
    --save_steps 200 \
    --output_dir "{MODEL_OUTPUT}" \
    --load_best_model_at_end True \
    --metric_for_best_model "eval_loss" \
    --greater_is_better False \
    --save_total_limit 2 \
    --disable_tqdm True \
    --report_to "none"

🚀 Starting FRESH training in ./pythia_70m_research_final for 5 epochs...
05/02/2026 04:10:41 - WARNING - __main__ - Process rank: -1, device: cuda:0, n_gpu: 1distributed training: False, 16-bits training: False
05/02/2026 04:10:41 - INFO - __main__ - Training/evaluation parameters TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import re
import nltk

# Download NLTK data for linguistic analysis
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

INFERENCE_MODEL_PATH = "./pythia_70m_research_final"

print(f"Loading Inference Engine from {INFERENCE_MODEL_PATH}...")
# Load the fine-tuned model and tokenizer for inference
tokenizer = AutoTokenizer.from_pretrained(INFERENCE_MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    INFERENCE_MODEL_PATH,
    torch_dtype=torch.float32
).to("cuda")

# Function to verify linguistic attributes in generated text
def verify_output(text):
    clean = re.sub(r'<\|.*?\|>', '', text).strip()
    sentences = [s for s in re.split(r'[.!?]+', clean) if len(s.strip()) > 3]
    words = nltk.word_tokenize(clean)
    tags = nltk.pos_tag(words)
    nouns = [w for w, t in tags if t.startswith('NN')]
    return len(sentences), len(nouns)

# Function to generate text using a given control token and prompt
def generate_text(token, prompt, max_len=50):
    full_prompt = f"{token} {prompt}"
    input_ids = tokenizer.encode(full_prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=max_len,
            do_sample=True,
            temperature=0.8,
            repetition_penalty=1.5,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_text = tokenizer.decode(output[0], skip_special_tokens=False)
    actual_sents, actual_nouns = verify_output(generated_text)

    print(f"\n--- Generation Result ---")
    print(f"Instruction Used: {token}")
    print(f"Full Prompt: '{prompt}'")
    print(f"Generated Output: \"{generated_text[len(full_prompt):].strip()}\"")
    print(f"Stats: {actual_sents} Sentences | {actual_nouns} Nouns")
    print("-" * 30)

# Test text generation with specific control tokens
generate_text(token="<|sents:1|> <|pos_noun:3|>", prompt="The diagnosis of the patient")
generate_text(token="<|sents:3|>", prompt="Once upon a time in the hospital")

Loading Inference Engine from ./pythia_70m_research_final...


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]


--- Generation Result ---
Instruction Used: <|sents:1|> <|pos_noun:3|>
Full Prompt: 'The diagnosis of the patient'
Generated Output: "is very positive, and will guide treatment for many years to come. This case includes three women who are on leave from a long-time life until she gets into it.<br /><hr /><|endoftext|>"
Stats: 3 Sentences | 17 Nouns
------------------------------

--- Generation Result ---
Instruction Used: <|sents:3|>
Full Prompt: 'Once upon a time in the hospital'
Generated Output: ", people would think it was only good on their part...and that is just an excuse to 'keep this movie cool'. (I'm not sure if I am going with my best friend or his new phone!)  It's always great when you"
Stats: 4 Sentences | 9 Nouns
------------------------------


In [ ]:
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import re
import nltk

MODEL_PATH = "./pythia_70m_research_final"
print(f"Loading model for final evaluation from {MODEL_PATH}...")

# Load the fine-tuned model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float32
).to("cuda")

# Function to calculate sentence count and noun density
def get_metrics(text):
    """
    Calculates sentence count and normalized noun density.
    """
    clean = re.sub(r'<\|.*?\|>', '', text).strip()
    words = nltk.word_tokenize(clean)

    sentences = [s for s in re.split(r'[.!?]+', clean) if len(s.strip()) > 3]

    if not words: return 0, 0
    tags = nltk.pos_tag(words)
    noun_count = len([w for w, t in tags if t.startswith('NN')])
    density = (noun_count / len(words)) * 100

    return len(sentences), density

# Function to check if a generated attribute matches the target bin
def is_successful(attr, actual_val, target_bin):
    """
    Validation logic using research-standard density thresholds.
    """
    if attr == "sents":
        return 1 if actual_val == target_bin else 0
    if attr == "NOUN":
        if target_bin == 1: return 1 if actual_val < 20 else 0
        if target_bin == 2: return 1 if 20 <= actual_val <= 35 else 0
        if target_bin == 3: return 1 if actual_val > 35 else 0
    return 0

# Define test cases for various linguistic attribute controls
test_cases = [
    {"token": "<|sents:1|>", "prompt": "The patient's condition is", "target": 1, "attr": "sents"},
    {"token": "<|sents:2|>", "prompt": "The new medical study indicates", "target": 2, "attr": "sents"},
    {"token": "<|sents:3|>", "prompt": "Symptoms of the flu include", "target": 3, "attr": "sents"},

    {"token": "<|pos_noun:1|>", "prompt": "It was very", "target": 1, "attr": "NOUN"},
    {"token": "<|pos_noun:2|>", "prompt": "The laboratory analysis of", "target": 2, "attr": "NOUN"},
    {"token": "<|pos_noun:3|>", "prompt": "Neurotransmitters in the synaptic", "target": 3, "attr": "NOUN"}
]

evaluation_data = []

print(f"\n{'='*30} GENERATED OUTPUTS {'='*30}")

# Iterate through test cases, generate text, and evaluate results
for case in test_cases:
    full_prompt = f"{case['token']} {case['prompt']}"
    inputs = tokenizer.encode(full_prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        output = model.generate(
            inputs,
            max_new_tokens=45,
            do_sample=True,
            temperature=0.7,
            repetition_penalty=1.6,
            pad_token_id=tokenizer.eos_token_id
        )

    full_text = tokenizer.decode(output[0], skip_special_tokens=False)
    generated_only = full_text[len(full_prompt):].strip()

    act_sents, act_density = get_metrics(full_text)
    actual_val = act_sents if case['attr'] == "sents" else act_density

    success = is_successful(case['attr'], actual_val, case['target'])
    evaluation_data.append({'Attribute': case['attr'], 'Target Bin': case['target'], 'Success': success})

    print(f"\n[INPUT]: {full_prompt}")
    print(f"[OUTPUT]: \"{generated_only}\"")
    print(f"[STATS]: {'Sentences' if case['attr'] == 'sents' else 'Noun Density'}: {actual_val:.1f} | Result: {'✔ MATCH' if success else '✘ FAIL'}")

# Display the final success matrix
df_results = pd.DataFrame(evaluation_data)
matrix = df_results.groupby(['Attribute', 'Target Bin'])['Success'].mean().unstack() * 100

print(f"\n{'='*30} FINAL SUCCESS MATRIX (%) {'='*30}")
print(matrix.fillna(0).round(2))

Loading model for final evaluation from ./pythia_70m_research_final...


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]


============================== GENERATED OUTPUTS ==============================

[INPUT]: <|sents:1|> The patient's condition is
[OUTPUT]: "not the worst of all but an awful kind.<|endoftext|>"
[STATS]: Sentences: 1.0 | Result: ✔ MATCH

[INPUT]: <|sents:2|> The new medical study indicates
[OUTPUT]: "that it's been a very good fit for the past couple of years and has made no attempt to provide any evidence supporting its claim.<|endoftext|>"
[STATS]: Sentences: 1.0 | Result: ✘ FAIL

[INPUT]: <|sents:3|> Symptoms of the flu include
[OUTPUT]: "fever, sore throat and swelling. After a few days (by 2pm), an MRI scan shows that there is no improvement in symptoms due to any symptom or signs other than headache/painming which does not cause this type"
[STATS]: Sentences: 2.0 | Result: ✘ FAIL

[INPUT]: <|pos_noun:1|> It was very
[OUTPUT]: "funny, everyone had been entertained by it. I remember this scene when i saw the movie "The Girl Who Loved You". We were just sitting in front of our backs

In [ ]:
import pandas as pd
import json
import re
import torch
import nltk
from transformers import AutoModelForCausalLM, AutoTokenizer

# Download necessary NLTK data
nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Define model path and training file path
MODEL_PATH = "./pythia_70m_trained_Final"
TRAIN_FILE = "./data/processed/train.json"

print("Setup complete. Environment initialized for linguistic evaluation.")

Setup complete. Environment initialized for linguistic evaluation.


In [ ]:
def get_actual_counts(text):
    """
    Analyzes the text to extract linguistic counts after removing control tokens.
    """
    clean_text = re.sub(r'<\|.*?\|>', '', text).strip()

    # Calculate sentence count
    sents_count = len([s for s in re.split(r'[.!?]+', clean_text) if len(s.strip()) > 3])

    # Calculate noun count
    words = nltk.word_tokenize(clean_text)
    tags = nltk.pos_tag(words)
    noun_count = len([word for word, tag in tags if tag.startswith('NN')])

    return sents_count, noun_count

print("Linguistic metric extractors defined.")

Linguistic metric extractors defined.


In [ ]:
def generate_table_4(file_path):
    """
    Parses the training file to generate statistical distributions of attributes.
    """
    data_list = []
    print(f"Reading training data from {file_path}...")

    with open(file_path, 'r') as f:
        for line in f:
            item = json.loads(line)
            text = item.get('text', "")

            # Extract control token values from the text
            s_match = re.search(r'<\|sents:(\d+)\|>', text)
            n_match = re.search(r'<\|pos_noun:(\d+)\|>', text)

            sents = int(s_match.group(1)) if s_match else 0
            nouns = int(n_match.group(1)) if n_match else 0

            # Calculate token count from clean text
            clean_text = re.sub(r'<\|.*?\|>', '', text).strip()
            tokens_count = len(clean_text.split())

            data_list.append({'sents': sents, 'NOUN': nouns, 'tokens': tokens_count})

    df = pd.DataFrame(data_list)

    # Generate descriptive statistics for the attributes
    stats = df.describe().T[['min', 'max', 'mean', 'std']]
    stats.columns = ['Min', 'Max', 'Avg', 'Std.']
    stats['# Bins'] = 3

    return stats

table_4 = generate_table_4(TRAIN_FILE)
print("\n--- TABLE 4: TRAINING DATA CHARACTERISTICS ---")
print(table_4)

Reading training data from ./data/processed/train.json...

--- TABLE 4: TRAINING DATA CHARACTERISTICS ---
        Min      Max         Avg        Std.  # Bins
sents   1.0      3.0    2.692132    0.614116       3
NOUN    1.0      3.0    2.770809    0.536569       3
tokens  5.0  27541.0  221.053812  312.960522       3


In [ ]:
print(f"Loading fine-tuned model: {MODEL_PATH}")
# Load fine-tuned model and tokenizer for evaluation
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float32
).to("cuda")

# Define test scenarios including control tokens, prompts, target values, and categories
test_scenarios = [
    ("<|sents:1|>", "The patient reported", 1, "sents"),
    ("<|sents:3|>", "The treatment plan", 3, "sents"),
    ("<|pos_noun:1|>", "The medicine was", 1, "nouns"),
    ("<|pos_noun:3|>", "Clinical observation showed", 5, "nouns")
]

print("Evaluation scenarios ready.")

Loading fine-tuned model: ./pythia_70m_trained_Final


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Evaluation scenarios ready.


In [ ]:
def run_evaluation_and_table_5(scenarios):
    """
    Performs inference and calculates the Success Rate for Table 5.
    """
    results_list = []
    print(f"\n{'='*20} QUANTITATIVE EVALUATION {'='*20}\n")

    for token, prompt, target, category in scenarios:
        input_text = f"{token} {prompt}"
        inputs = tokenizer.encode(input_text, return_tensors="pt").to("cuda")

        with torch.no_grad():
            output = model.generate(
                inputs,
                max_new_tokens=45,
                do_sample=True,
                temperature=0.7,
                repetition_penalty=1.5,
                pad_token_id=tokenizer.eos_token_id
            )

        full_output = tokenizer.decode(output[0], skip_special_tokens=False)
        actual_sents, actual_nouns = get_actual_counts(full_output)

        # Determine success based on attribute category
        if category == "sents":
            is_success = (actual_sents == target)
            attr_label = "sents"
        else:
            is_success = (actual_nouns >= target)
            attr_label = "NOUN"

        results_list.append({'Attribute': attr_label, 'Success': is_success})

        print(f"Instruction: {token} | Goal: {target} | Observed: {actual_sents if category=='sents' else actual_nouns} | Result: {'✔' if is_success else '✘'}")

    # Create a DataFrame from results and calculate success rates
    df_results = pd.DataFrame(results_list)
    performance = df_results.groupby('Attribute')['Success'].apply(lambda x: (x.sum()/len(x))*100).reset_index()
    performance.columns = ['Attribute', 'Success Rate (%)']

    # Map attributes to categories for organized display
    category_map = {'sents': 'Lexical', 'NOUN': 'POS Tags'}
    performance['Category'] = performance['Attribute'].map(category_map)

    return performance.sort_values(['Category', 'Attribute']).set_index(['Category', 'Attribute'])

table_5 = run_evaluation_and_table_5(test_scenarios)
print("\n--- TABLE 5: ATTRIBUTE CONTROL PERFORMANCE ---")
print(table_5)


==================== QUANTITATIVE EVALUATION ====================

Instruction: <|sents:1|> | Goal: 1 | Observed: 2 | Result: ✘
Instruction: <|sents:3|> | Goal: 3 | Observed: 2 | Result: ✘
Instruction: <|pos_noun:1|> | Goal: 1 | Observed: 10 | Result: ✔
Instruction: <|pos_noun:3|> | Goal: 5 | Observed: 13 | Result: ✔

--- TABLE 5: ATTRIBUTE CONTROL PERFORMANCE ---
                    Success Rate (%)
Category Attribute                  
Lexical  sents                   0.0
POS Tags NOUN                  100.0


In [ ]:
!zip -r colab_ready_project.zip \
    data/processed/ \
    pythia_70m_research_final/ \
    FINAL_NLP_updateeed_\(1\).ipynb \
    run_clm.py \
    -x "**/checkpoint-*" "**/optimizer.pt" "**/scheduler.pt"

updating: data/processed/ (stored 0%)
updating: data/processed/master_binned.json (deflated 60%)
updating: data/processed/test.jsonl (deflated 60%)
updating: data/processed/train.json (deflated 60%)
updating: data/processed/val.json (deflated 60%)
updating: data/processed/train_with_tokens.json (deflated 60%)
updating: pythia_70m_research_final/ (stored 0%)
updating: pythia_70m_research_final/config.json (deflated 50%)
updating: pythia_70m_research_final/generation_config.json (deflated 29%)
updating: pythia_70m_research_final/model.safetensors (deflated 9%)
updating: pythia_70m_research_final/tokenizer_config.json (deflated 46%)
updating: pythia_70m_research_final/tokenizer.json (deflated 82%)
updating: pythia_70m_research_final/training_args.bin (deflated 51%)
updating: pythia_70m_research_final/train_results.json (deflated 41%)
updating: pythia_70m_research_final/all_results.json (deflated 55%)
updating: pythia_70m_research_final/trainer_state.json (deflated 78%)
updating: pythia_70